> 📄 English: [cognee-ai-memory.ipynb](cognee-ai-memory.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--ai-memory-with-cognee--cognee-ai-memory)

# 使用 Cognee 构建智能 AI 记忆系统：一个 Python 开发知识图谱

本教程演示如何构建一个智能知识系统，将权威的 Python 实践、社区标准和个人开发上下文统一到一个连贯的 AI 记忆层中。通过将 Python 之父 Guido van Rossum 的贡献与既定的设计原则以及你自己的开发模式连接起来，你将创建一个知识图谱，为 Python 开发生成上下文相关、可解释且一致的洞察。

## 理解架构

现代软件开发涉及在多个事实来源之间穿梭：语言文档、社区最佳实践、历史代码示例以及个人经验。传统方法把这些当作相互独立的资源，需要人工综合和解读。本教程演示了一种使用知识图谱和检索增强生成（RAG）的根本不同方法。

我们要构建的系统把分散的开发数据转化为一个相互关联的知识网络。你不必在孤立的文档中搜索，而是查询一个统一的图，它理解 Python 的设计哲学、其创造者的真实实现模式，以及你具体的开发上下文之间的关系。这样就能发现非显而易见的联系，比如你遇到的某个类型注解难题与 Guido 在 mypy 中实现的解决方案有何关联，或者你的 async 模式如何与 Python 的核心设计原则相一致。

这种方法的力量在于它能够从数据中推断隐式关系和规则。通过高级图算法和 AI 处理，系统能识别跨不同数据来源的模式，从而创建一个随着每次交互而变得越来越智能的记忆层。

## 系统架构与数据流


# <div align="center">
#   <img src="./data/cognee_diagram.png" alt="System Architecture and Data Flow" width="700"/>
# </div>


## Cognee 的核心操作

Cognee 框架提供四个基本操作，将原始数据转化为智能知识系统。每个操作在知识图谱流水线中都有特定的用途。

**add()** 函数是数据摄入的入口，接受 JSON、Markdown 和 API 响应等多种格式。该操作会规范化并准备数据，以供图谱构建。**cognify()** 函数代表核心转换引擎，使用 AI 提取实体、识别关系，并将数据结构化为可遍历的知识图谱。通过 **search()**，用户可以使用自然语言查询或结构化的 Cypher 表达式与图谱交互，系统会理解上下文和关系并返回相关结果。最后，**memify()** 应用高级算法从数据中推断隐式连接和规则，创建一个动态记忆层，增强搜索能力并发现非显而易见的模式。

这些操作协同工作，构建出的系统不仅存储信息，还能以有意义的方式理解并连接信息。

## 数据来源及其作用

本教程利用一组精心整理的数据来源，代表 Python 开发的不同视角。每个来源都为知识图谱贡献独特的价值。

**guido_contributions.json** 文件包含 Guido van Rossum 在 mypy 和 CPython 上的真实 pull request 和 commit。它们提供了 Python 之父如何进行语言设计和问题求解的权威示例。**pep_style_guide.md** 编码了社区认可的 Python 代码风格和类型约定标准，确保生成的洞察与既定的最佳实践一致。**zen_principles.md** 则捕捉了 Python 的哲学基础，让技术决策扎根于简洁、显式和可读性原则。

个人上下文来自 **my_developer_rules.md**（包含项目特定的约定和约束）以及 **copilot_conversations.json**（保存了真实的开发对话，包括问题、代码片段和讨论主题）。这种组合创建了一个从语言哲学到实际实现无所不包的全面知识库。

## 环境配置

Cognee 使用异步函数来高效处理复杂的图操作。以下配置确保其在 Jupyter notebook 环境中正确执行。

### 启用异步执行

配置 notebook 环境，以支持 Cognee 图处理引擎所需的 async/await 操作。

In [ ]:
%pip install cognee==0.3.3

In [ ]:
import nest_asyncio
nest_asyncio.apply()

## 模型配置

为了在处理速度、成本效率和输出质量之间取得最佳平衡，本教程使用 OpenAI 的 GPT-4o-mini 模型。请确保你的环境配置文件（.env）包含以下设置：

```
LLM_MODEL="gpt-4o-mini"
```

该模型为实体提取和关系映射提供了足够的能力，同时为教学目的保持合理的处理时间。

### 验证 Cognee 安装

确认 Cognee 已正确安装，并判断你使用的是本地开发版本还是已安装的包。

In [ ]:
import cognee
import os
from pathlib import Path

print('Quick Cognee Import Check')
print('=' * 30)
print(f'Cognee location: {cognee.__file__}')
print(f'Package directory: {os.path.dirname(cognee.__file__)}')

# Check if it's local or installed
current_dir = Path.cwd()
cognee_path = Path(cognee.__file__)
if current_dir in cognee_path.parents:
    print('Status: LOCAL DEVELOPMENT VERSION')
else:
    print('Status: INSTALLED PACKAGE')

### 配置 Python 路径

通过将项目根目录添加到系统路径，确保 Python 解释器能够找到所有必要的模块。

In [ ]:
import sys
from pathlib import Path
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
else:
    project_root = Path.cwd()

# Add project root to the beginning of sys.path
project_root_str = str(project_root.absolute())
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Project root: {project_root_str}")

### 初始化干净状态

移除任何现有的 Cognee 数据，确保教程从一个全新的知识图谱开始。

In [ ]:
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)

## 构建知识图谱：Guido 的贡献

知识图谱的构建从摄入 Guido van Rossum 的开发历史开始。该数据集包含他在 mypy 和 CPython 上的详细 pull request 和 commit，为语言设计决策和实现模式提供了具体示例。

摄入过程包含两个关键步骤。首先，add() 函数加载原始 JSON 数据，并将其分配给一个命名的节点集以便组织。然后，cognify() 处理这些数据，提取实体、识别关系，并在不同贡献之间建立时间关联。temporal_cognify 参数启用了基于时间的分析，允许查询 Python 特性随时间的演变。

### 加载并处理 Guido 的开发数据

摄入贡献数据，并构建具备时间感知能力的初始知识图谱。

In [ ]:
import cognee

result = await cognee.add(
    os.path.abspath("data/guido_contributions.json"),
    node_set=["guido_data"]
)
await cognee.cognify(temporal_cognify=True)

### 检查初始搜索结果

显示第一个搜索结果，以验证图谱构建成功。

In [ ]:
results = await cognee.search("Show me commits")
print(results[0])

## 通过可视化理解图谱结构

搜索操作展示出与传统数据库查询的根本不同。Cognee 不是检索孤立的记录，而是遍历一个知识图谱，该图谱理解 commit、语言特性、设计决策及其随时间演变之间的关系。这样就能发现通过传统搜索方法难以识别的模式和联系。

可视化能让我们深入了解图谱的结构以及 Cognee 已识别出的关系。交互式图谱揭示出围绕不同项目和时间段的聚类模式，展示想法如何演变为特性，以及不同贡献之间如何相互关联。

### 生成交互式图谱可视化

创建知识图谱结构的 HTML 可视化。

In [ ]:
from cognee import visualize_graph
await visualize_graph('./guido_contributions.html')

### 显示知识图谱

在 notebook 中渲染交互式可视化，以便探索。

In [ ]:
from IPython.display import IFrame, HTML, display
display(IFrame("./guido_contributions.html", width="100%", height="500"))

In [ ]:
# Alternative method to open the graph directly in your default browser, if the IFrame doesn't work
import os
import webbrowser
from pathlib import Path

html_path = Path('guido_contributions.html').resolve()
file_url = html_path.as_uri()

print(f"HTML file path: {html_path}")
print(f"Opening: {file_url}")

webbrowser.open(file_url)


## 图谱分析与模式识别

可视化揭示了 Python 发展历史中的几个重要模式。CPython 核心开发在 2020 年前后表现出集中的活动，而 mypy 的贡献则特别聚焦于 fixtures 和 run 类。PEP 讨论创建了与 Thomas Grainger、Adam Turner 等其他贡献者的联系，展现了语言演化的协作本质。基于时间的联系展示了最初的想法和讨论如何演变成具体的特性和实现。

这些可视化模式不仅帮助我们理解做了哪些改动，还能理解 Python 开发的不同方面如何相互关联。图谱结构使追踪想法从最初提案到实现再到完善的演化过程成为可能。

可视化的交互性允许更深入的探索。你可以查看特定聚类以理解专注的开发工作，或追踪看似无关的贡献之间的联系，从而发现 Python 演化中隐藏的模式。

## 扩展知识图谱

在通过 Guido 的贡献建立基础之后，下一阶段是集成更多数据来源，以创建一个全面的知识系统。这次扩展将权威示例与社区标准、设计哲学和个人开发上下文连接起来。

每个数据来源都被分配到一个特定的节点集，从而在图内形成逻辑分组。包括对话和个人规则在内的开发者专属数据形成一个聚类，而原则和指南形成另一个聚类。这种组织方式既能在特定领域内进行有针对性的搜索，又能保持整个图谱的连接。

### 摄入完整数据集

添加所有剩余的数据来源，并将它们处理到统一的知识图谱中。

In [ ]:
import os

await cognee.add(os.path.abspath("data/copilot_conversations.json"), node_set=["developer_data"])
await cognee.add(os.path.abspath("data/my_developer_rules.md"), node_set=["developer_data"])
await cognee.add(os.path.abspath("data/zen_principles.md"), node_set=["principles_data"])
await cognee.add(os.path.abspath("data/pep_style_guide.md"), node_set=["principles_data"])

await cognee.cognify(temporal_cognify=True)

### 发现跨领域联系

查询扩展后的图谱，寻找个人开发难题与 Guido 的解决方案之间的联系。

In [ ]:
results = await cognee.search(
    "What validation issues did I encounter in January 2024, and how would they be addressed in Guido's contributions?",
    query_type=cognee.SearchType.GRAPH_COMPLETION
)
print(results)

## 分析关联的洞察

搜索结果展示了 Cognee 如何连接不同的数据来源以提供上下文洞察。系统识别你的开发难题与 Guido 的历史解决方案之间的模式，揭示诸如类型注解中的循环导入问题（这正对应 Guido 在某些 mypy pull request 中解决过的问题），或者列表推导式的性能优化（遵循 CPython commit 中确立的模式）等联系。

这些联系超越了简单的关键词匹配。图谱理解不同概念之间的语义关系，使其能够识别出：你遇到的类型注解难题在概念上与 mypy 中的工作相关，即使具体术语有所不同。

## 高级记忆层构建

memify 操作代表了 Cognee 知识处理中最复杂的一面。该函数对现有图谱应用高级算法，推断跨不同数据来源的隐式规则和模式。与基础图谱构建不同，memify 创建了一个动态记忆层，它不仅理解存在哪些信息，还理解不同信息之间如何相互关联和影响。

记忆层使系统能够识别一些模式，例如 Guido 的贡献中反复出现的、与特定 Zen 原则相一致的设计决策，或者针对特定类型 Python 难题的常见解决策略。这形成了一种「机构记忆」，能够基于历史模式和既定的最佳实践提供指导。

### 构建智能记忆层

应用高级模式识别和规则推断算法来创建记忆层。

In [ ]:
await cognee.memify()

## 记忆层能力

memify 操作为知识图谱增强了几项针对 Python 开发的复杂能力。它从示例中推断编码模式，识别某些方法何时会在相似上下文中反复出现。系统将抽象的设计哲学与具体的实现决策连接起来，把「显式优于隐式」等原则与类型注解或 API 设计中的具体编码选择相关联。

这形成了一个反馈循环：系统既从权威来源学习，也从实践经验中学习，不断完善它对有效 Python 开发模式的理解。

### 分析设计模式的一致性

查询经过记忆增强的图谱，理解个人实现如何与 Python 哲学保持一致。

In [ ]:
# Search for connections between your async patterns and Python philosophy
results = await cognee.search(
    query_text= "How does my AsyncWebScraper implementation align with Python's design principles?",
    query_type=cognee.SearchType.GRAPH_COMPLETION
)
print("Python Pattern Analysis:", results)

## 使用节点集过滤进行定向搜索

数据摄入期间建立的节点集组织方式，使在知识图谱的特定领域内进行精确、定向的搜索成为可能。当寻求特定主题的权威指导时，这种过滤能力尤其有价值，因为它让查询聚焦于相关数据来源，同时保持对更广泛上下文的感知。

通过将搜索限定在特定节点集，你可以确保回复来自合适的来源。关于风格指南的问题可以直接指向 PEP 文档和设计原则，而实现问题则可以聚焦于实际代码示例和开发者经验。

### 在特定知识领域内搜索

通过仅查询原则和指南节点集，演示定向搜索。

In [ ]:
from cognee.modules.engine.models.node_set import NodeSet
results = await cognee.search(
    query_text= "How should variables be named?",
    query_type=cognee.SearchType.GRAPH_COMPLETION,
    node_type=NodeSet,
    node_name=['principles_data']
)

## 时间分析能力

图谱构建期间启用的 temporal cognify 选项，为理解 Python 开发如何随时间演变提供了强大的能力。这种时间感知能力允许查询探索趋势、识别开发活动密集的时期，或理解特定特性如何出现并成熟。

时间查询可以揭示关于开发速度、在相似时间范围内引入的不同特性之间的关系，或实现方法如何随语言演进而变化的洞察。这种历史视角为理解当前最佳实践和预测未来方向提供了宝贵的上下文。

### 探索基于时间的开发模式

查询知识图谱的时间维度，以理解开发的演变。

In [ ]:
await cognee.search(
    query_text = "What can we learn from Guido's contributions in 2025?",
    query_type=cognee.SearchType.TEMPORAL
)

## 通过反馈实现持续学习

知识图谱系统通过一个反馈机制支持持续改进，该机制捕获搜索结果的实用性和相关性。这就创建了一个随着时间推移适应你特定需求和偏好的学习系统。

当搜索交互被保存时，系统可以跟踪哪些类型的查询最常见、哪些结果最有用，以及你的开发重点如何演变。这些反馈会成为图谱本身的一部分，影响未来的搜索，帮助系统提供越来越相关的结果。

### 执行带反馈跟踪的搜索

执行一次启用交互保存的搜索，以支持后续的反馈。

In [ ]:
answer = await cognee.search(
    query_type=cognee.SearchType.GRAPH_COMPLETION,
    query_text="What is the most zen thing about Python?",
    save_interaction=True,  # This enables feedback later
)

### 提供反馈以增强未来的搜索

提交关于搜索结果的反馈，以改进系统对你偏好的理解。

In [ ]:
feedback = await cognee.search(
    query_type=cognee.SearchType.FEEDBACK,
    query_text="Last result was useful, I like code that complies with best practices.",
    last_k=1,
)

## 结论与下一步

本教程演示了如何构建一个智能知识系统，将 Python 开发的多个视角统一到一个连贯、可查询的图谱中。你构建的系统把 Python 之父的权威示例与社区标准、设计哲学和个人开发经验连接起来，为 Python 开发中的明智决策创建了一个强大的工具。

知识图谱方法与传统的文档和搜索方法相比具有显著优势。通过理解不同信息之间的关系，系统能提供用传统方式难以甚至不可能发现的洞察。时间感知、模式识别和反馈机制共同创建了一个随时间变得越来越有价值的动态系统。

随着你继续使用和扩展这个系统，可以考虑添加更多数据来源，例如项目文档、代码审查或技术讨论。每个新数据来源都会丰富图谱，创建更多连接并支持更复杂的洞察。反馈循环确保系统适应你的特定需求，成为你 Python 开发旅程中越来越有价值的伙伴。